<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-12-ethics-and-responsible-ai/lesson-12.4-privacy-and-data-protection/notebooks/exercises-12_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 12.4: Privacy and Data Protection

*Module 12 · Ethics, Safety, Governance & Explainability*

> LLMs process text that often contains personal information. This lesson builds a PII detection pipeline (regex + NER), an anonymization system with reversible fake data, and covers GDPR compliance for AI — the practical infrastructure for data protection.

`PII Detection` · `Anonymization` · `GDPR` · `Regex + NER` · `75 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-12.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — PII Detection Pipeline — Regex + NER — `01_pii_detector.py`
2. Step 2 — Anonymization Pipeline — Replace PII with Fake Data — `02_anonymization.py`
3. Step 3 — GDPR Compliance for AI Systems — `03_gdpr_checks.py`


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · PII Detection Pipeline — Regex + NER

Find emails, phones, Aadhaar, PAN, names, addresses in text automatically.

**`01_pii_detector.py`** · _Block 1 of 3_

PII DETECTION PIPELINE — Regex + NER


In [ ]:
# PII DETECTION PIPELINE — Regex + NER
import re
from dataclasses import dataclass

@dataclass
class PIIMatch:
    type: str; value: str; start: int; end: int

class PIIDetector:
    PATTERNS = {
        "email": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        "phone_in": r"\b(?:\+91[\s-]?)?[6-9]\d{9}\b",
        "aadhaar": r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
        "pan": r"\b[A-Z]{5}\d{4}[A-Z]\b",
        "credit_card": r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
        "ip_address": r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b",
    }

    def detect(self, text):
        matches = []
        for pii_type, pattern in self.PATTERNS.items():
            for m in re.finditer(pattern, text):
                matches.append(PIIMatch(pii_type, m.group(), m.start(), m.end()))
        return matches

    def redact(self, text):
        for m in sorted(self.detect(text), key=lambda x:-x.start):
            text = text[:m.start] + f"[{m.type.upper()}]" + text[m.end:]
        return text

detector = PIIDetector()
text = "Contact Priya at [email protected] or 9876543210. PAN: ABCDE1234F. Aadhaar: 1234 5678 9012."
print("PII Detection:\n")
for m in detector.detect(text):
    print(f"  {m.type:15s}: {m.value}")
print(f"\n  Redacted: {detector.redact(text)}")


> **What just happened?** 6 PII types detected automatically: email, phone, Aadhaar, PAN, credit card, IP. The redact() method replaces each with [EMAIL] , [AADHAAR] , etc. Run this on EVERY input before sending to an LLM and on EVERY output before showing to users. Two-way PII protection.


### Step 2 · Anonymization Pipeline — Replace PII with Fake Data

Detect, replace with realistic fakes, process, then de-anonymize.

**`02_anonymization.py`** · _Block 2 of 3_

ANONYMIZATION PIPELINE — Replace PII with realistic fakes


In [ ]:
# ANONYMIZATION PIPELINE — Replace PII with realistic fakes
import re, uuid, hashlib

class Anonymizer:
    \"\"\"Replace PII with consistent fake values. Reversible.\"\"\"
    def __init__(self):
        self.mapping = {}  # real → fake
        self.reverse = {}  # fake → real

    def _fake(self, pii_type, real_value):
        key = f"{pii_type}:{real_value}"
        if key not in self.mapping:
            h = hashlib.md5(real_value.encode()).hexdigest()[:8]
            fakes = {"email":f"user_{h}@example.com", "phone":f"9000{h[:6]}",
                     "name":f"Person_{h[:4]}"}
            fake = fakes.get(pii_type, f"[{pii_type}_{h}]")
            self.mapping[key] = fake
            self.reverse[fake] = real_value
        return self.mapping[key]

    def anonymize(self, text):
        for m in re.finditer(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", text):
            text = text.replace(m.group(), self._fake("email", m.group()))
        return text

    def deanonymize(self, text):
        for fake, real in self.reverse.items():
            text = text.replace(fake, real)
        return text

anon = Anonymizer()
original = "Email [email protected] about the refund for [email protected]"
anonymized = anon.anonymize(original)
restored = anon.deanonymize(anonymized)
print("Anonymization Pipeline:\n")
print(f"  Original:    {original}")
print(f"  Anonymized:  {anonymized}")
print(f"  Restored:    {restored}")
print("\n  Pattern: anonymize → send to LLM → get response → deanonymize")


### Step 3 · GDPR Compliance for AI Systems

Data subject rights, lawful basis, retention policies coded as checks.

**`03_gdpr_checks.py`** · _Block 3 of 3_

GDPR COMPLIANCE CHECKS FOR AI SYSTEMS


In [ ]:
# GDPR COMPLIANCE CHECKS FOR AI SYSTEMS

GDPR_REQUIREMENTS = {
    "Lawful basis": ["Consent obtained for data processing",
                      "Legitimate interest documented",
                      "Purpose limitation: data used only for stated purpose"],
    "Data subject rights": ["Right to access: user can export their data",
                              "Right to deletion: user can request data removal",
                              "Right to rectification: user can correct their data",
                              "Right to explanation: automated decisions can be explained"],
    "Data protection": ["PII encrypted at rest and in transit",
                          "Access logging for all PII access",
                          "Data retention policy: delete after N days",
                          "Data processing agreements with LLM providers"],
    "AI-specific": ["No PII in prompts sent to third-party LLMs",
                      "Training data does not contain EU personal data without consent",
                      "Automated decision opt-out available"],
}

print("GDPR Compliance for AI:\n")
total = 0
for cat, items in GDPR_REQUIREMENTS.items():
    print(f"  {cat}:")
    for item in items:
        total += 1; print(f"    [ ] {item}")
print(f"\n  {total} requirements. Complete for GDPR compliance.")


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
